# SkinGen: AI-Powered Skincare Recommendation System

## Project Overview

implements an intelligent recommendation system for skincare products that prioritizes both **relevance** and **safety**. The system combines content-based filtering with domain specific safety rules to provide personalized product recommendations.

### Key Features:
- Content-based filtering using cosine similarity
- Hybrid safety approach (hard filtering + soft penalties) -> rule based
- Multi-metric evaluation (NDCG, Precision, Recall, Safety)
- Cross-validation with hyperparameter optimization

- **Python**: scikit-learn, pandas, numpy
- **ML Techniques**: Content-based filtering, feature engineering for ML, hyperparameter tuning
- **Evaluation**: 5-fold cross-validation, multiple ranking metrics


## Part 1: Data Loading & Setup

This project uses:
- **10,300+- skincare products** with ingredient list and concern data
- **200 synthetic test queries** representing realistic user requests

### Decision: Why Synthetic Queries?
Real user data is unavailable, so I generated synthetic queries that:
- Match actual product concern distributions
- Include realistic combinations (e.g., "dry skin" + "hydrating" concerns)
- Provide reproducible evaluation data


In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances, manhattan_distances
from sklearn.metrics import ndcg_score
from sklearn.model_selection import train_test_split, ParameterGrid, KFold
import json


# Load data
products = pd.read_parquet('../data/cleaned/skingen_products_lean_clean.parquet')

with open('../data/synthetic_test_queries.json', 'r') as f:
    all_queries = json.load(f)

print(f'Loaded {len(products):,} products')
print(f'Loaded {len(all_queries)} test queries')

Loaded 10,296 products
Loaded 200 test queries


## Part 2: Feature Engineering

Machine learning models work with numerical vectors, not text. I need to convert product attributes (concerns, ingredients) into numerical features that can capture similarity.

### Decision: One-Hot Encoding + Count Features
I chose a **hybrid feature representation**:

1. **Binary features (one-hot encoding)**: Captures which concerns a product addresses
   - Example: `[1, 0, 1, ...]` means product has "brightening" and "anti-aging" but not "hydrating"
   
2. **Count features (normalized)**: Captures how many concerns/ingredients
   - Higher counts = more comprehensive formulation
   - Normalized to 0-1 range for fair comparison

### Why?
- **Simple and interpretable**: Easy to understand which features drive recommendations
- **Captures both presence and quantity**: "Has concern" + "How many concerns"

### Alternative Considered:
- **TF-IDF**: Better for text, but our concerns are already categorical
- **Word embeddings**: Overkill for 11 well-defined concerns


In [3]:
# Fix NaN arrays
def fix_nan(arr):
    return arr if isinstance(arr, np.ndarray) else np.array([])

products['positive_concerns'] = products['positive_concerns'].apply(fix_nan)
products['negative_concerns'] = products['negative_concerns'].apply(fix_nan)
products['condition_concerns'] = products['condition_concerns'].apply(fix_nan)


In [4]:
# One-hot encoding
mlb_pos = MultiLabelBinarizer()
mlb_neg = MultiLabelBinarizer()
mlb_cond = MultiLabelBinarizer()

pos_encoded = mlb_pos.fit_transform(products['positive_concerns'])
neg_encoded = mlb_neg.fit_transform(products['negative_concerns'])
cond_encoded = mlb_cond.fit_transform(products['condition_concerns'])

# Normalize counts
pos_norm = products['positive_count'] / products['positive_count'].max()
neg_norm = products['negative_count'] / products['negative_count'].max()
cond_norm = products['condition_count'] / products['condition_count'].max()



In [5]:
# Build feature matrix
def build_features(count_weight=0.0):
    """Build feature matrix with optional count weighting"""
    return np.hstack([
        pos_encoded,
        neg_encoded,
        cond_encoded,
        pos_norm.values.reshape(-1, 1) * count_weight,
        neg_norm.values.reshape(-1, 1) * count_weight,
        cond_norm.values.reshape(-1, 1) * count_weight
    ])

# Build initial features
features = build_features(count_weight=0.0)

In [6]:
print(f'Feature matrix: {features.shape}')
print(f'- Positive concerns: {len(mlb_pos.classes_)}')
print(f'- Negative concerns: {len(mlb_neg.classes_)}')
print(f'- Condition concerns: {len(mlb_cond.classes_)}')

Feature matrix: (10296, 20)
- Positive concerns: 11
- Negative concerns: 4
- Condition concerns: 2


## Part 3: Safety Rules

### Why Safety Rules?
Skincare is not just about relevance : **safety is critical**. Recommending a "brightening" product that causes severe dryness for dry skin users is not optimal.

### Domain Knowledge Integration
After analyzing the dataset, I found:
- **48% of products contain "drying" ingredients**
- **64% contain "irritating" ingredients**
- **24% may worsen oily skin**

This means a **pure similarity-based system would frequently recommend unsafe products**.

### Decision: Two-Layer Safety System

**Layer 1: Hard Filters (Absolute No-Go)**
- Remove products that conflict with user's skin conditions
- Example: Never recommend eczema-triggering products to eczema users

**Layer 2: Soft Penalties (Skin Type Conflicts)**
- Downrank (don't remove) products with skin type mismatches
- Example: Penalize "drying" products for dry skin, but keep in pool if highly relevant

### Why This Hybrid Approach?
- **Hard filter**: Prevents dangerous recommendations (medical safety)
- **Soft penalty**: Balances safety with recommendation quality (user satisfaction)
- **Flexibility**: Penalty strength is tunable (becomes a hyperparameter)
- => user could have been ; dry skin , looking for anti-aging or acne fighting : products could have labels with ''drying'' in it... using soft penalty will still give us this product ; but lower compared to product without it. Since the concern can have actives that causes drying : Retinol/retinoids/AHA , BHA acids ... So we will soft penalty it and warn when needed. So user can make informed decision ; and or patch test product.

### Alternative Considered:
- **Hard filter only**: Too restrictive, many queries would return 0 results
- **Soft penalty only**: Not safe enough for medical domain for eczema & rosacea users.


In [7]:
# Concern-based hard filters (Layer 1: Hard) Very contradictive ruling
CONCERN_RULES = {
    'redness_reducing': ['irritating'],
    'reduces_irritation': ['irritating'],
    'hydrating': ['drying'],
    'soothing': ['irritating'],
}
CONDITION_WARNING_TRIGGERS = { # ( warning for skincondition users)
    'rosacea': ['irritating', 'drying'],
    'eczema': ['irritating', 'drying']
}
# Skin type penalties (Layer 2: Soft)
SKIN_TYPE_PENALTY_RULES = {
    'dry_skin': ['drying'],
    'oily_skin': ['may_worsen_oily_skin'],
    'sensitive_skin': ['irritating'],
    'combination_skin': ['drying', 'may_worsen_oily_skin']
}


## Part 4: Recommendation System

### Core Algorithm: Hybrid Content-Based Filtering

The recommendation pipeline has 6 steps:

#### Step 1: Product Type Filtering
Filter candidates by requested type (serum, moisturizer, etc.)

#### Step 2: Hard Safety Filter
Remove products with **absolute contraindications**:
- Concern conflicts (e.g., "irritating" for redness-reducing queries)
- Condition conflicts (e.g., eczema-triggering for eczema users)

#### Step 3: Similarity Calculation
Use **cosine similarity** between user query vector and product feature vectors.

**Why Cosine Similarity?**
- high-dimensional sparse features
- Range: 0 (no similarity) to 1 (identical)
- Invariant to feature scaling
- Computationally efficient

#### Step 4: Soft Penalty Application
Multiply similarity scores by penalty factor (0.0-1.0) for skin type mismatches.

**Example:** Product with 0.95 similarity but "drying" for dry skin -> 0.95 x 0.8 = 0.76

#### Step 5: Ranking
Sort by penalized similarity scores, return top N.

#### Step 6: Warning Generation
Add contextual warnings for 'negative tag claimed' products that made it through.

### Decision:
**Collaborative filtering** requires:
- User-item interaction history (ratings, purchases)
- Multiple users for pattern detection

**I don't have this data**, so content-based filtering is the right choice:
- Works with product attributes alone
- No cold-start problem
- Explainable recommendations ("matched because of brightening + hydrating")

### Hyperparameters:
- `penalty_value`: Strength of soft penalty (0.8-1.0)
- `count_weight`: Weight given to concern counts vs binary presence (0.0-0.3)


In [8]:
def create_query_vector(user_query, count_weight=0.0):
    """Create feature vector from user query"""
    query = np.zeros(features.shape[1])
    
    user_concerns = user_query.get('concerns', [])
    if user_concerns:
        valid = [c for c in user_concerns if c in mlb_pos.classes_]
        if valid:
            pos_vec = mlb_pos.transform([valid])[0]
            query[0:len(mlb_pos.classes_)] = pos_vec
    
    query[-4] = count_weight
    query[-1] = count_weight #* 0.25
    
    return query


def get_avoid_list(user_query):
    """Get concerns to avoid based on user query"""
    avoid = set()
    for concern in user_query.get('concerns', []):
        if concern in CONCERN_RULES:
            avoid.update(CONCERN_RULES[concern])
    return avoid


def get_warnings(row, user_query):
    """Generate warnings for potentially risky products"""
    warnings = []
    neg = row['negative_concerns']
    product_type = row['type'].lower()
    
    if not isinstance(neg, np.ndarray):
        return warnings
    
    skin_type = user_query.get('skin_type')
    user_conditions = user_query.get('skin_conditions', [])
    
    is_moisturizer = 'moisturizer' in product_type or 'cream' in product_type
    
    # Skin type warnings
    if skin_type == 'dry_skin':
        if 'drying' in neg:
            if is_moisturizer:
                warnings.append(f'Info: [{skin_type}] May cause dryness')
            else:
                warnings.append(f'Info: [{skin_type}] May cause dryness - use with moisturizer')
    
    if skin_type == 'oily_skin':
        if 'may_worsen_oily_skin' in neg:
            warnings.append(f'Info: [{skin_type}] May increase oiliness')
    
    if skin_type == 'sensitive_skin':
        if 'irritating' in neg:
            warnings.append(f'Caution: [{skin_type}] May irritate - patch test recommended')
        if 'drying' in neg:
            warnings.append(f'Info: [{skin_type}] May cause dryness')
    
    if skin_type == 'combination_skin':
        if 'drying' in neg:
            warnings.append(f'Info: [{skin_type}] May cause dryness in dry areas')
        if 'may_worsen_oily_skin' in neg:
            warnings.append(f'Info: [{skin_type}] May increase oiliness in T-zone')
    
    # Condition warnings
    for user_cond in user_conditions:
        if user_cond in ['rosacea', 'eczema']:
            if 'irritating' in neg:
                warnings.append(f'Caution: [{user_cond}] May trigger {user_cond} - patch test recommended')
    
    return warnings


def recommend_hybrid(user_query, penalty_value=0.8, n=10):
    """
    Hybrid Content-Based Recommendation System
    
    Combines hard filtering (removes dangerous products) with
    soft penalties (downranks borderline cases) for safe and
    relevant skincare recommendations.
    """
    # 1. Filter by product type
    product_type = user_query.get('product_type', 'all')
    if product_type != 'all':
        candidates = products[products['type'] == product_type].copy()
    else:
        candidates = products.copy()
    
    if len(candidates) == 0:
        return pd.DataFrame()
    
    # 2. HARD FILTER: Remove products with concern conflicts
    avoid = get_avoid_list(user_query)
    user_conditions = user_query.get('skin_conditions', [])
    
    def is_safe(row):
        neg = row['negative_concerns']
        if isinstance(neg, np.ndarray):
            for concern in avoid:
                if concern in neg:
                    return False
        
        cond = row['condition_concerns']
        if isinstance(cond, np.ndarray):
            for condition in user_conditions:
                if condition in cond:
                    return False
        return True
    
    candidates = candidates[candidates.apply(is_safe, axis=1)]
    
    if len(candidates) == 0:
        return pd.DataFrame()
    
    # 3. Calculate cosine similarity
    candidate_indices = candidates.index.tolist()
    candidate_features = features[candidate_indices]
    query_vector = create_query_vector(user_query)
    similarities = cosine_similarity(query_vector.reshape(1, -1), candidate_features)[0]
    
    # 4. SOFT PENALTY: Cumulative downranking for skin type conflicts
    skin_type = user_query.get('skin_type')
    if skin_type and skin_type in SKIN_TYPE_PENALTY_RULES:
        penalty_tags = SKIN_TYPE_PENALTY_RULES[skin_type]
        
        for i, product_idx in enumerate(candidate_indices):
            product = products.loc[product_idx]
            neg = product['negative_concerns']
            
            if isinstance(neg, np.ndarray):
                # Count matching penalty tags (cumulative approach)
                penalty_count = sum(1 for tag in penalty_tags if tag in neg)
                
                if penalty_count > 0:
                    # Compound penalty: each violation multiplies
                    similarities[i] *= (penalty_value ** penalty_count)
    # 5. Get top N
    top_n = min(n, len(candidates))
    top_indices = np.argsort(similarities)[::-1][:top_n]
    
    results = candidates.iloc[top_indices].copy()
    results['score'] = similarities[top_indices]
    results['rank'] = range(1, len(results) + 1)
    
    # 6. Add warnings
    warnings_list = []
    for idx, row in results.iterrows():
        warnings_list.append(get_warnings(row, user_query))
    results['warnings'] = warnings_list
    
    return results


## Part 5: Hyperparameter Optimization & Evaluation

### Evaluation Strategy

#### 1. Train/Test Split
Split 200 queries into:
- **Training set (60%)**: Used for hyperparameter optimization with 5-fold cross-validation
- **Test set (40%)**: Held out for final unbiased evaluation

This prevents overfitting and validates that the model generalizes to unseen queries.

#### 2. Hyperparameter Grid Search
**Goal:** Find optimal `penalty` and `count_weight`

**Search Space:**
- `penalty`: [0.8, 0.9, 1.0] (3 values)
- `count_weight`: [0.0, 0.1, 0.2, 0.3] (4 values)
- **Total: 12 combinations**

**Method:** Grid search with 5-fold cross-validation on training set

#### 3. Metrics
- **NDCG@10**: Ranking quality (0-1, higher = better order)
- **Precision@5**: Relevance of top-5 recommendations
- **Recall@5**: Coverage of user concerns in top-5
- **Safety Rate**: Percentage of recommendations without skin type conflicts


In [9]:
# Split data: 60% train, 40% test
train_queries, test_queries = train_test_split(all_queries, test_size=0.4, random_state=42)
print(f'Training set: {len(train_queries)} queries (60%)')
print(f'Test set: {len(test_queries)} queries (40%)')


Training set: 120 queries (60%)
Test set: 80 queries (40%)


### Hyperparameter Optimization + Cross-Validation


In [10]:
#NDCG + Safety

# Helpers
def eval_params(queries, penalty, weight):
    """Evaluate NDCG for given parameters"""
    global features
    features = build_features(count_weight=weight)
    
    scores = []
    for query in queries:
        user_wants = set(query['concerns'])
        recs = recommend_hybrid(query, penalty_value=penalty, n=10)
        
        if len(recs) == 0:
            continue
        
        # Calculate relevance scores
        relevance = []
        for idx, rec in recs.iterrows():
            prod_concerns = rec['positive_concerns']
            overlap = len(user_wants & set(prod_concerns)) if isinstance(prod_concerns, np.ndarray) else 0
            relevance.append(overlap)
        
        if sum(relevance) > 0:
            scores.append(ndcg_score([relevance], [list(recs['score'])], k=10))
    
    return np.mean(scores) if scores else 0.0


def eval_safety(queries, penalty):
    """Evaluate safety rate for given penalty"""
    safe_count = 0
    total_count = 0
    
    for query in queries:
        skin_type = query.get('skin_type')
        if not skin_type:
            continue
        
        recs = recommend_hybrid(query, penalty_value=penalty, n=5)
        
        for idx, rec in recs.iterrows():
            total_count += 1
            neg_concerns = rec['negative_concerns']
            
            # Check if safe for this skin type
            is_safe = True
            if isinstance(neg_concerns, np.ndarray):
                if skin_type == 'dry_skin' and 'drying' in neg_concerns:
                    is_safe = False
                elif skin_type == 'oily_skin' and 'may_worsen_oily_skin' in neg_concerns:
                    is_safe = False
                elif skin_type == 'sensitive_skin' and 'irritating' in neg_concerns:
                    is_safe = False
            
            if is_safe:
                safe_count += 1
    
    return safe_count / total_count if total_count > 0 else 0

# Grid search
param_grid = {'penalty': [0.8, 0.9, 1.0], 'weight': [0.0, 0.1, 0.2, 0.3]}
kf = KFold(n_splits=5, shuffle=True, random_state=42)

print(f'Testing {len(list(ParameterGrid(param_grid)))} combinations\n')

results = []
for p in ParameterGrid(param_grid):
    print(f"p={p['penalty']:.1f}, w={p['weight']:.1f}...", end=' ')
    
    cv = [eval_params([train_queries[i] for i in val], p['penalty'], p['weight'])
          for _, val in kf.split(train_queries)]
    
    features = build_features(count_weight=p['weight'])
    safety = eval_safety(train_queries, p['penalty'])
    
    results.append({**p, 'ndcg': np.mean(cv), 'std': np.std(cv), 'safety': safety})
    print(f"NDCG={np.mean(cv):.4f}±{np.std(cv):.4f}, Safety={safety:.1%}")

# Convert to DataFrame and show results
df = pd.DataFrame(results)
print('RESULTS:')
print(df.sort_values('ndcg', ascending=False))

# Pick best: safety >= 92%, max NDCG (or fallback to top-3 NDCG -> max safety)
print('BEST PARAMETERS:')
best = (df[df['safety'] >= 0.92].nlargest(1, 'ndcg') 
        if (df['safety'] >= 0.92).any() 
        else df.nlargest(3, 'ndcg').nlargest(1, 'safety')).iloc[0]

print(f"p={best['penalty']:.1f}, w={best['weight']:.1f} | " +
      f"NDCG={best['ndcg']:.4f}±{best['std']:.4f} | Safety={best['safety']:.1%}")

# Penalty comparison
print('Penalty Trade-off:')
print(df.groupby('penalty')[['ndcg', 'safety']].max().to_string())

Testing 12 combinations

p=0.8, w=0.0... NDCG=0.9670±0.0146, Safety=95.1%
p=0.8, w=0.1... NDCG=0.9648±0.0155, Safety=95.1%
p=0.8, w=0.2... NDCG=0.9648±0.0155, Safety=95.1%
p=0.8, w=0.3... NDCG=0.9648±0.0155, Safety=95.1%
p=0.9, w=0.0... NDCG=0.9672±0.0143, Safety=94.3%
p=0.9, w=0.1... NDCG=0.9650±0.0154, Safety=94.3%
p=0.9, w=0.2... NDCG=0.9650±0.0154, Safety=94.3%
p=0.9, w=0.3... NDCG=0.9650±0.0154, Safety=94.3%
p=1.0, w=0.0... NDCG=0.9672±0.0153, Safety=90.4%
p=1.0, w=0.1... NDCG=0.9655±0.0156, Safety=89.9%
p=1.0, w=0.2... NDCG=0.9655±0.0156, Safety=90.1%
p=1.0, w=0.3... NDCG=0.9655±0.0156, Safety=90.1%
RESULTS:
    penalty  weight      ndcg       std    safety
8       1.0     0.0  0.967236  0.015310  0.903896
4       0.9     0.0  0.967203  0.014334  0.942857
0       0.8     0.0  0.967015  0.014564  0.950649
9       1.0     0.1  0.965531  0.015619  0.898701
11      1.0     0.3  0.965531  0.015619  0.901299
10      1.0     0.2  0.965531  0.015619  0.901299
5       0.9     0.1  0.96500

### Detailed Evaluation with Best Parameters


In [11]:
print(f"Using: penalty={best['penalty']:.1f}, weight={best['weight']:.1f}\n")

features = build_features(count_weight=best['weight'])

def eval_query(query, penalty):
    """Evaluate single query - returns dict of metrics"""
    user_wants = set(query['concerns'])
    recs = recommend_hybrid(query, penalty_value=penalty, n=10)
    
    if len(recs) == 0:
        return None
    
    # Calc in 1 pass
    metrics = {'ndcg': 0, 'prec': 0, 'rec': 0, 'safe': 0, 'total': 0}
    relevance, matched = [], set()
    
    for i, (_, row) in enumerate(recs.iterrows()):
        prod_concerns = set(row['positive_concerns']) if isinstance(row['positive_concerns'], np.ndarray) else set()
        overlap = len(user_wants & prod_concerns)
        relevance.append(overlap)
        
        if i < 5:  # Top-5 only
            if overlap > 0:
                metrics['prec'] += 1
            matched.update(user_wants & prod_concerns)
            
            # Safety check
            if query.get('skin_type'):
                metrics['total'] += 1
                neg = row['negative_concerns']
                skin = query['skin_type']
                
                safe = True
                if isinstance(neg, np.ndarray):
                    safe = not ((skin == 'dry_skin' and 'drying' in neg) or
                               (skin == 'oily_skin' and 'may_worsen_oily_skin' in neg) or
                               (skin == 'sensitive_skin' and 'irritating' in neg))
                
                if safe:
                    metrics['safe'] += 1
    
    # Finalize metrics
    if sum(relevance) > 0:
        metrics['ndcg'] = ndcg_score([relevance], [list(recs['score'])], k=10)
    metrics['prec'] /= 5
    metrics['rec'] = len(matched) / len(user_wants) if user_wants else 0
    
    return metrics

# 5-fold CV
kf = KFold(n_splits=5, shuffle=True, random_state=42)
all_results = []

for fold, (_, val_idx) in enumerate(kf.split(train_queries), 1):
    print(f'Fold {fold}/5...', end=' ')
    
    fold_metrics = [eval_query(train_queries[i], best['penalty']) for i in val_idx]
    fold_metrics = [m for m in fold_metrics if m is not None]
    
    # Aggregate fold
    ndcg = np.mean([m['ndcg'] for m in fold_metrics])
    prec = np.mean([m['prec'] for m in fold_metrics])
    rec = np.mean([m['rec'] for m in fold_metrics])
    
    total_recs = sum(m['total'] for m in fold_metrics)
    safe_recs = sum(m['safe'] for m in fold_metrics)
    safety = safe_recs / total_recs if total_recs > 0 else 0
    
    all_results.append({'ndcg': ndcg, 'prec': prec, 'rec': rec, 'safety': safety})
    print(f'NDCG={ndcg:.4f} P@5={prec:.4f} R@5={rec:.4f} Safety={safety:.1%}')


df = pd.DataFrame(all_results)
#final results
print(df.describe().loc[['mean', 'std', 'min', 'max']].T.round(4))
print(f"penalty={best['penalty']:.1f}, weight={best['weight']:.1f} | " +
      f"NDCG: {df['ndcg'].mean():.4f}+-{df['ndcg'].std():.4f} | Safety: {df['safety'].mean():.1%}")

Using: penalty=0.9, weight=0.0

Fold 1/5... NDCG=0.9809 P@5=1.0000 R@5=0.9514 Safety=88.6%
Fold 2/5... NDCG=0.9619 P@5=1.0000 R@5=0.9583 Safety=93.3%
Fold 3/5... NDCG=0.9859 P@5=1.0000 R@5=0.9653 Safety=98.6%
Fold 4/5... NDCG=0.9607 P@5=1.0000 R@5=0.9653 Safety=96.2%
Fold 5/5... NDCG=0.9467 P@5=1.0000 R@5=0.9514 Safety=94.4%
          mean     std     min     max
ndcg    0.9672  0.0160  0.9467  0.9859
prec    1.0000  0.0000  1.0000  1.0000
rec     0.9583  0.0069  0.9514  0.9653
safety  0.9423  0.0373  0.8857  0.9857
penalty=0.9, weight=0.0 | NDCG: 0.9672+-0.0160 | Safety: 94.2%


### Test Set Evaluation


In [12]:
print(f'Evaluating on {len(test_queries)} held-out test queries')
print(f'Using best parameters: penalty={best["penalty"]:.1f}, weight={best["weight"]:.1f}\n')

# Rebuild features with best params
features = build_features(count_weight=best['weight'])

# Evaluate on test set
test_metrics = [eval_query(q, best['penalty']) for q in test_queries]
test_metrics = [m for m in test_metrics if m is not None]

# Aggregate
test_ndcg = np.mean([m['ndcg'] for m in test_metrics])
test_prec = np.mean([m['prec'] for m in test_metrics])
test_rec = np.mean([m['rec'] for m in test_metrics])

total_test = sum(m['total'] for m in test_metrics)
safe_test = sum(m['safe'] for m in test_metrics)
test_safety = safe_test / total_test if total_test > 0 else 0

print('Test Set Results:')
print(f'  NDCG@10:      {test_ndcg:.4f}')
print(f'  Precision@5:  {test_prec:.4f}')
print(f'  Recall@5:     {test_rec:.4f}')
print(f'  Safety:       {test_safety:.1%}')


print('TRAINING vs TEST COMPARISON')

print(f'{"Metric":<15} {"Training (CV)":<20} {"Test Set"}')

print(f'{"NDCG@10":<15} {df["ndcg"].mean():.4f} ± {df["ndcg"].std():.4f}      {test_ndcg:.4f}')
print(f'{"Precision@5":<15} {df["prec"].mean():.4f} ± {df["prec"].std():.4f}      {test_prec:.4f}')
print(f'{"Recall@5":<15} {df["rec"].mean():.4f} ± {df["rec"].std():.4f}      {test_rec:.4f}')
print(f'{"Safety":<15} {df["safety"].mean():.1%} ± {df["safety"].std():.1%}        {test_safety:.1%}')


print(f'Test performance: NDCG={test_ndcg:.4f}, Safety={test_safety:.1%}')


Evaluating on 80 held-out test queries
Using best parameters: penalty=0.9, weight=0.0

Test Set Results:
  NDCG@10:      0.9779
  Precision@5:  1.0000
  Recall@5:     0.9667
  Safety:       92.8%
TRAINING vs TEST COMPARISON
Metric          Training (CV)        Test Set
NDCG@10         0.9672 ± 0.0160      0.9779
Precision@5     1.0000 ± 0.0000      1.0000
Recall@5        0.9583 ± 0.0069      0.9667
Safety          94.2% ± 3.7%        92.8%
Test performance: NDCG=0.9779, Safety=92.8%


## Part 6: Demo - System in Action (Optimized)

Now that we've found the best parameters, let's see the system in action with real queries.


In [15]:
#SYSTEM DEMO (Optimized Parameters)
print(f"Using: penalty={best['penalty']:.1f}, weight={best['weight']:.1f}\n")

# Rebuild features with best weight
features = build_features(count_weight=best['weight'])

# demo ; SAMPLE RECOMMENDATION


sample_query = {
    'concerns': ['brightening', 'hydrating', 'anti_aging'],
    'skin_type': 'dry_skin',
    'product_type': 'serum',
    'skin_conditions': []
}

print(f'Query:')
print(f'  Concerns: {sample_query["concerns"]}')
print(f'  Skin Type: {sample_query["skin_type"]}')
print(f'  Product Type: {sample_query["product_type"]}')

results = recommend_hybrid(sample_query, penalty_value=0.8, n=5)

print(f'Top 5 Recommendations:')
print(results[['brand', 'name', 'type', 'score']].to_string(index=False))

if len(results) > 0:
    first_warnings = results.iloc[0]['warnings']
    if first_warnings:
        print(f'Warnings for top product:')
        for warning in first_warnings:
            print(f'  {warning}')
    else:
        print(f'Top product has no warnings for this user')

Using: penalty=0.9, weight=0.0

Query:
  Concerns: ['brightening', 'hydrating', 'anti_aging']
  Skin Type: dry_skin
  Product Type: serum
Top 5 Recommendations:
             brand                                 name  type    score
             Boots         Ingredients Collagen Booster serum 0.816497
          Mediheal      N.M.F Intensive Hydrating Serum serum 0.816497
Advanced Clinicals Hyaluronic Acid Hydrating Face Serum serum 0.816497
          Skin Inc                       Collagen Serum serum 0.816497
            boscia         Vegan Collagen Booster Serum serum 0.816497
Top product has no warnings for this user


## Part 7: Model Validation

### Distance Metric Comparison

We validate our choice of cosine similarity by comparing it to alternative distance metrics.

**Key difference:**
- **Cosine similarity**: Outputs similarity score (0-1, higher = more similar)
- **Euclidean/Manhattan distance**: Outputs distance (0-∞, higher = more different)

To compare fairly, we convert distance to similarity: `similarity = 1/(1+distance)`
- Distance = 0 → similarity = 1 (identical)
- Distance → ∞ → similarity → 0 (very different)


In [14]:
#DISTANCE METRIC COMPARISON (Validation)
#Comparing distance metrics on test set with best parameters

# Rebuild features with best weight
features = build_features(count_weight=best['weight'])

# Test each metric
results = {}
for metric in ['cosine', 'euclidean', 'manhattan']:
    scores = []
    
    for query in test_queries[:50]:  # Sample for speed
        wants = set(query['concerns'])
        ptype = query.get('product_type')
        candidates = products[products['type'] == ptype]
        
        if len(candidates) == 0:
            continue
        
        # Get query vector and candidate features
        q_vec = create_query_vector(query, count_weight=best['weight'])
        c_feat = features[candidates.index]
        
        # Calculate similarity with chosen metric
        if metric == 'cosine':
            sims = cosine_similarity(q_vec.reshape(1, -1), c_feat)[0]
        elif metric == 'euclidean':
            # Convert distance -> similarity (1/(1+d): distance 0->sim 1, distance ∞->sim 0)
            sims = 1 / (1 + euclidean_distances(q_vec.reshape(1, -1), c_feat)[0])
        else:
            # Convert distance -> similarity (same formula as euclidean)
            sims = 1 / (1 + manhattan_distances(q_vec.reshape(1, -1), c_feat)[0])
        
        # Get top-10 relevance
        top_10 = np.argsort(sims)[::-1][:10]
        rel = [len(wants & set(candidates.iloc[i]['positive_concerns'])) 
               if isinstance(candidates.iloc[i]['positive_concerns'], np.ndarray) else 0 
               for i in top_10]
        
        if sum(rel) > 0:
            scores.append(ndcg_score([rel], [sims[top_10]], k=10))
    
    results[metric] = np.mean(scores) if scores else 0

# Display results
print(f'{"Metric":<12} {"NDCG@10"}')

for metric in ['cosine', 'euclidean', 'manhattan']:
    marker = ' <- Selected' if metric == 'cosine' else ''
    print(f'{metric:<12} {results[metric]:.4f}{marker}')

Metric       NDCG@10
cosine       0.9800 <- Selected
euclidean    0.8861
manhattan    0.8861


## Conclusion & Next Steps

### Results Summary

The hybrid recommendation system achieved excellent performance:

- **NDCG@10: ~0.97** (97% of perfect ranking quality)
- **Precision@5: ~1.00** (100% of recommendations are relevant)
- **Recall@5: ~0.96** (96% of user needs covered)
- **Safety: ~94%** (only 6% potentially unsafe recommendations)

### Key Findings

1. **Penalty value matters**: Lower penalties (0.8-0.9) significantly improve safety with minimal impact on relevance
2. **Count features don't help**: weight=0.0 performs best, suggesting binary presence/absence is more important than quantity
3. **Cosine similarity is optimal**: Outperforms euclidean and manhattan distance for this task
4. **Model generalizes well**: Similar performance on training and test sets indicates no overfitting

### Limitations & Future Improvements

**Current System Limitations:**
- **Basic concern-level matching**: System relies on product-level concern labels (e.g., "hydrating", "irritating") rather than ingredient-level properties
- **No ingredient-specific filtering**: Cannot filter based on specific ingredients (e.g., user wants "no fragrance" but system can't identify fragrance components in ingredient lists)
- **Trust in product labels**: Recommendations depend on manufacturer-provided concern labels, which may be incomplete or inconsistent

**Improvement 1: Ingredient-Level Dataset Mapping**
- Map each ingredient to its properties (comedogenic rating, irritation potential, fragrance vs functional, concentration effects)
- Enable granular filtering: "recommend products WITHOUT any fragrance components" or "WITH niacinamide at effective concentrations"
- Move from basic product-label recommendations to precise ingredient-aware recommendations
- Example: Currently blocks products labeled "irritating"; with ingredient mapping, could block products containing specific irritants like "Alcohol Denat" or "Essential Oils"

**Improvement 2: Granular User Control**
- **Concern importance weighting**: Let users prioritize concerns (e.g., "hydration is critical, brightening is nice-to-have")
- **Ingredient blacklist/whitelist**: User specifies "must have retinol" or "no parabens"
- **Preference learning**: Adapt recommendations based on user feedback and product history

**Improvement 3: System Enhancements**
- **Diversity constraints**: Avoid recommending 5 similar products with same active ingredients
- **Explainability**: Show why each product was recommended ("Contains niacinamide for brightening, alcohol-free for sensitive skin")
- **Context-aware filtering**: Consider product combinations (e.g., recommend complementary morning/night routines)


### Next Steps

**Immediate (Thesis Scope):**
1. **Deploy as C# ML.NET API**: Implement current concern-based recommendation system in production environment
2. **Validate label-based recommender**: Ensure the system works reliably with product-level concern labels
3. **Test with synthetic queries**: Verify performance matches Python prototype results

**Future Enhancements:**

4. **Build ingredient property database**: Map 4,000+ ingredients to properties (comedogenic ratings, irritation levels, functional categories)
5. **Implement ingredient-level filtering**: Enable precise "no fragrance", "with retinol", "silicone-free" recommendations
6. **Add user preference controls**: Allow concern weighting and ingredient inclusion/exclusion rules
7. **Validate with real users**: Collect feedback to refine ingredient mappings and concern labels